# Phase 3: Provider & Fiber Feature Engineering

**Goal:** Clean the FCC BDC fiber data for over-reporting artifacts, then extract fiber presence, provider counts, major provider flags, and spatial neighbor features for each CB and filing period.

**Inputs:**
- FCC BDC data for 3 filing periods (Jun 2022, Jun 2023, Jun 2024)
- Adjacency table from Phase 1
- Provider master set

**Output:** `cb_provider_features.parquet`

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Paths
INPUT_DIR = '/Users/yugeshchandraroy/Downloads/Projects/market-demographics-fcc-pipeline/output'
OUTPUT_DIR = '/Users/yugeshchandraroy/Downloads/Projects/broadband-market-scanner/data/intermediate'

FCC_FILES = {
    '2022-06': f'{INPUT_DIR}/all_states_block_level_availability_202206.csv',
    '2023-06': f'{INPUT_DIR}/all_states_block_level_availability_202306.csv',
    '2024-06': f'{INPUT_DIR}/all_states_block_level_availability_202406.csv',
}

ADJACENCY_FILE = f'{INPUT_DIR}/adjacency.parquet'

print('Phase 3: Provider & Fiber Feature Engineering')
print('=' * 55)

Phase 3: Provider & Fiber Feature Engineering


## Step 1: Load FCC BDC Data (All Filing Periods)

In [2]:
# Load all filing periods into a single DataFrame
frames = []
for period, filepath in FCC_FILES.items():
    print(f'Loading {period}...')
    df = pd.read_csv(filepath, dtype={'block_geoid': str, 'provider_id': str, 'state_fips': str})
    # Ensure filing_period is set correctly
    df['filing_period'] = period
    frames.append(df)
    print(f'  Rows: {len(df):,}  |  Unique blocks: {df["block_geoid"].nunique():,}  |  Tech codes: {sorted(df["technology"].unique())}')

fcc = pd.concat(frames, ignore_index=True)
print(f'\nCombined: {len(fcc):,} rows  |  {fcc["block_geoid"].nunique():,} unique blocks  |  {fcc["filing_period"].nunique()} periods')
print(f'Filing periods: {sorted(fcc["filing_period"].unique())}')
print(f'Technology codes: {sorted(fcc["technology"].unique())}')

Loading 2022-06...
  Rows: 840,512  |  Unique blocks: 720,672  |  Tech codes: [np.int64(50)]
Loading 2023-06...
  Rows: 1,071,161  |  Unique blocks: 860,560  |  Tech codes: [np.int64(50)]
Loading 2024-06...
  Rows: 1,172,724  |  Unique blocks: 921,036  |  Tech codes: [np.int64(50)]

Combined: 3,084,397 rows  |  969,554 unique blocks  |  3 periods
Filing periods: ['2022-06', '2023-06', '2024-06']
Technology codes: [np.int64(50)]


## Step 2: Compute Raw `has_fiber` Flag Per CB Per Period

In [3]:
# Fiber = technology code 50
fcc['is_fiber'] = (fcc['technology'] == 50).astype(int)

# Raw has_fiber: 1 if ANY record for this (block, period) has tech code 50
raw_fiber = fcc.groupby(['filing_period', 'block_geoid'])['is_fiber'].max().reset_index()
raw_fiber.rename(columns={'is_fiber': 'has_fiber_raw'}, inplace=True)

print(f'Raw fiber flags computed: {len(raw_fiber):,} (block, period) combinations')
print(f'\nRaw fiber presence by period:')
for period in sorted(raw_fiber['filing_period'].unique()):
    subset = raw_fiber[raw_fiber['filing_period'] == period]
    fiber_count = subset['has_fiber_raw'].sum()
    total = len(subset)
    print(f'  {period}: {fiber_count:,} / {total:,} blocks with fiber ({100*fiber_count/total:.1f}%)')

Raw fiber flags computed: 2,502,268 (block, period) combinations

Raw fiber presence by period:
  2022-06: 720,672 / 720,672 blocks with fiber (100.0%)
  2023-06: 860,560 / 860,560 blocks with fiber (100.0%)
  2024-06: 921,036 / 921,036 blocks with fiber (100.0%)


## Step 3: Forward-Persistence Correction for Over-Reported Fiber

Providers over-report fiber coverage. Since physical fiber doesn't get removed, a CB showing fiber in an earlier filing but not in a later one was over-reported.

**Rule:** Walk backward from the latest filing. Find the earliest period from which `has_fiber_raw = 1` persists unbroken through to the latest filing. Set `has_fiber = 1` only from that point onward.

In [4]:
# Get sorted filing periods
periods_sorted = sorted(raw_fiber['filing_period'].unique())
print(f'Filing periods (chronological): {periods_sorted}')

# Pivot: rows=block_geoid, columns=filing_period, values=has_fiber_raw
# First, get all unique blocks across ALL periods
all_blocks = raw_fiber['block_geoid'].unique()
print(f'Total unique blocks across all periods: {len(all_blocks):,}')

fiber_pivot = raw_fiber.pivot(index='block_geoid', columns='filing_period', values='has_fiber_raw')
# Fill NaN with 0 — if a block doesn't appear in a period, it has no fiber (and no service)
fiber_pivot = fiber_pivot.fillna(0).astype(int)
# Ensure columns are in chronological order
fiber_pivot = fiber_pivot[periods_sorted]

print(f'Pivot shape: {fiber_pivot.shape}')
print(f'\nSample timelines (first 10 blocks with any fiber):')
sample = fiber_pivot[fiber_pivot.sum(axis=1) > 0].head(10)
print(sample)

Filing periods (chronological): ['2022-06', '2023-06', '2024-06']
Total unique blocks across all periods: 969,554
Pivot shape: (969554, 3)

Sample timelines (first 10 blocks with any fiber):
filing_period    2022-06  2023-06  2024-06
block_geoid                               
060014001001010        1        1        1
060014001001011        1        1        1
060014001001013        1        1        1
060014001001014        1        1        1
060014001001015        1        1        1
060014001001016        1        1        1
060014001001017        1        1        1
060014001001018        1        1        1
060014001001019        1        1        1
060014001001020        1        1        1


In [5]:
# Apply forward-persistence correction
# Walk backward from the latest filing.
# Find the earliest period from which has_fiber=1 persists UNBROKEN through the latest.

raw_values = fiber_pivot.values  # numpy array for speed
corrected = np.zeros_like(raw_values)
n_periods = raw_values.shape[1]

# For each block (row), walk backward from the last column
# Find the earliest index where fiber=1 persists unbroken to the end
for col_idx in range(n_periods - 1, -1, -1):
    if col_idx == n_periods - 1:
        # Last period: corrected = raw (accept as-is)
        corrected[:, col_idx] = raw_values[:, col_idx]
    else:
        # A block has fiber at this period only if:
        # 1. It has fiber_raw=1 at this period, AND
        # 2. It has corrected fiber=1 at the NEXT period (persistence through to latest)
        corrected[:, col_idx] = raw_values[:, col_idx] & corrected[:, col_idx + 1]

fiber_corrected = pd.DataFrame(corrected, index=fiber_pivot.index, columns=fiber_pivot.columns)

# Summary of corrections
flipped = (raw_values == 1) & (corrected == 0)
total_flipped = flipped.sum()
total_raw_fiber = (raw_values == 1).sum()

print('Forward-Persistence Correction Summary')
print('=' * 50)
print(f'Total (block, period) entries flipped from 1→0: {total_flipped:,}')
print(f'Total raw fiber entries: {total_raw_fiber:,}')
print(f'Percentage corrected: {100*total_flipped/total_raw_fiber:.2f}%')

print(f'\nCorrections by period:')
for i, period in enumerate(periods_sorted):
    period_flipped = flipped[:, i].sum()
    period_raw = (raw_values[:, i] == 1).sum()
    print(f'  {period}: {period_flipped:,} flipped / {period_raw:,} raw fiber ({100*period_flipped/max(period_raw,1):.1f}%)')

print(f'\nCorrected fiber presence by period:')
for i, period in enumerate(periods_sorted):
    corrected_count = corrected[:, i].sum()
    total = corrected.shape[0]
    print(f'  {period}: {corrected_count:,} / {total:,} blocks ({100*corrected_count/total:.1f}%)')

Forward-Persistence Correction Summary
Total (block, period) entries flipped from 1→0: 70,097
Total raw fiber entries: 2,502,268
Percentage corrected: 2.80%

Corrections by period:
  2022-06: 34,776 flipped / 720,672 raw fiber (4.8%)
  2023-06: 35,321 flipped / 860,560 raw fiber (4.1%)
  2024-06: 0 flipped / 921,036 raw fiber (0.0%)

Corrected fiber presence by period:
  2022-06: 685,896 / 969,554 blocks (70.7%)
  2023-06: 825,239 / 969,554 blocks (85.1%)
  2024-06: 921,036 / 969,554 blocks (95.0%)


In [6]:
# Unpivot corrected fiber back to long format
fiber_long = fiber_corrected.reset_index().melt(
    id_vars='block_geoid',
    var_name='filing_period',
    value_name='has_fiber'
)
print(f'Corrected fiber long format: {len(fiber_long):,} rows')
print(fiber_long.head())

Corrected fiber long format: 2,908,662 rows
       block_geoid filing_period  has_fiber
0  060014001001010       2022-06          1
1  060014001001011       2022-06          1
2  060014001001013       2022-06          1
3  060014001001014       2022-06          1
4  060014001001015       2022-06          1


## Step 4: Compute Fiber & Provider Features Per CB Per Period

In [7]:
# Fiber-specific records
fiber_records = fcc[fcc['technology'] == 50].copy()

# Compute per (filing_period, block_geoid):
# - fiber_provider_count: distinct providers with fiber
# - max_fiber_down, max_fiber_up: max speeds among fiber records
fiber_stats = fiber_records.groupby(['filing_period', 'block_geoid']).agg(
    fiber_provider_count_raw=('provider_id', 'nunique'),
    max_fiber_down=('max_download', 'max'),
    max_fiber_up=('max_upload', 'max'),
).reset_index()

# Total provider count (all technologies)
total_providers = fcc.groupby(['filing_period', 'block_geoid']).agg(
    total_provider_count=('provider_id', 'nunique')
).reset_index()

# Merge fiber stats with total provider count
provider_features = total_providers.merge(
    fiber_stats, on=['filing_period', 'block_geoid'], how='left'
)
provider_features['fiber_provider_count_raw'] = provider_features['fiber_provider_count_raw'].fillna(0).astype(int)

# Merge with corrected has_fiber
provider_features = provider_features.merge(
    fiber_long, on=['filing_period', 'block_geoid'], how='left'
)
provider_features['has_fiber'] = provider_features['has_fiber'].fillna(0).astype(int)

# Apply correction: if has_fiber was corrected to 0, zero out fiber-specific fields
mask_no_fiber = provider_features['has_fiber'] == 0
provider_features.loc[mask_no_fiber, 'fiber_provider_count_raw'] = 0
provider_features.loc[mask_no_fiber, 'max_fiber_down'] = 0
provider_features.loc[mask_no_fiber, 'max_fiber_up'] = 0

provider_features.rename(columns={'fiber_provider_count_raw': 'fiber_provider_count'}, inplace=True)

print(f'Provider features: {len(provider_features):,} rows')
print(f'\nSample:')
print(provider_features.head(10))

Provider features: 2,502,268 rows

Sample:
  filing_period      block_geoid  total_provider_count  fiber_provider_count  \
0       2022-06  060014001001010                     2                     2   
1       2022-06  060014001001011                     1                     1   
2       2022-06  060014001001013                     2                     2   
3       2022-06  060014001001014                     1                     1   
4       2022-06  060014001001015                     1                     1   
5       2022-06  060014001001016                     1                     1   
6       2022-06  060014001001017                     1                     1   
7       2022-06  060014001001018                     2                     2   
8       2022-06  060014001001019                     1                     1   
9       2022-06  060014001001020                     1                     1   

   max_fiber_down  max_fiber_up  has_fiber  
0            1000          1000

## Step 5: Major Provider One-Hot Flags

Define major ISPs and generate presence/fiber binary flags per CB per period.

In [8]:
# Major provider definitions: short_name -> (holding_company_pattern, provider_ids)
# Using holding_company matching since provider_id is consistent across periods
MAJOR_PROVIDERS = {
    'att':       {'pattern': 'AT&T Inc.', 'id': '130077'},
    'charter':   {'pattern': 'Charter Communications', 'id': '130235'},
    'comcast':   {'pattern': 'Comcast Corporation', 'id': '130317'},
    'verizon':   {'pattern': 'Verizon Communications', 'id': '131425'},
    'frontier':  {'pattern': 'Frontier Communications', 'id': '130258'},
    'cox':       {'pattern': 'Cox Communications', 'id': '130360'},
    'google_fiber': {'pattern': 'Google LLC', 'id': '240041'},
    'altice':    {'pattern': 'Altice USA', 'id': '130370'},
    'windstream': {'pattern': 'Windstream', 'id': '131413'},
    'lumen':     {'pattern': 'Lumen Technologies', 'id': '130228'},
}

print(f'Major providers defined: {len(MAJOR_PROVIDERS)}')
for name, info in MAJOR_PROVIDERS.items():
    count = fcc[fcc['holding_company'].str.contains(info['pattern'], case=False, na=False)].shape[0]
    print(f'  {name}: {count:,} records across all periods (pattern: "{info["pattern"]}")')

Major providers defined: 10
  att: 855,597 records across all periods (pattern: "AT&T Inc.")
  charter: 233,023 records across all periods (pattern: "Charter Communications")
  comcast: 32,047 records across all periods (pattern: "Comcast Corporation")
  verizon: 279,256 records across all periods (pattern: "Verizon Communications")
  frontier: 306,553 records across all periods (pattern: "Frontier Communications")
  cox: 8,124 records across all periods (pattern: "Cox Communications")
  google_fiber: 35,964 records across all periods (pattern: "Google LLC")
  altice: 151,474 records across all periods (pattern: "Altice USA")
  windstream: 50,096 records across all periods (pattern: "Windstream")
  lumen: 970 records across all periods (pattern: "Lumen Technologies")


In [9]:
# Build one-hot flags for each major provider
# For each (filing_period, block_geoid), check if provider is present (any tech) and present with fiber (tech 50)

major_flags_list = []

for name, info in MAJOR_PROVIDERS.items():
    print(f'Processing {name}...')
    # Filter records for this provider
    provider_records = fcc[fcc['holding_company'].str.contains(info['pattern'], case=False, na=False)]
    
    # Presence flag (any technology)
    presence = provider_records.groupby(['filing_period', 'block_geoid']).size().reset_index(name='_count')
    presence[f'{name}_present'] = 1
    presence = presence[['filing_period', 'block_geoid', f'{name}_present']]
    
    # Fiber flag (tech code 50 only)
    fiber_recs = provider_records[provider_records['technology'] == 50]
    fiber_presence = fiber_recs.groupby(['filing_period', 'block_geoid']).size().reset_index(name='_count')
    fiber_presence[f'{name}_fiber'] = 1
    fiber_presence = fiber_presence[['filing_period', 'block_geoid', f'{name}_fiber']]
    
    major_flags_list.append(('presence', name, presence))
    major_flags_list.append(('fiber', name, fiber_presence))

print('Done computing raw flags.')

Processing att...
Processing charter...
Processing comcast...
Processing verizon...
Processing frontier...
Processing cox...
Processing google_fiber...
Processing altice...
Processing windstream...
Processing lumen...
Done computing raw flags.


In [10]:
# Merge all major provider flags into provider_features
for flag_type, name, flag_df in major_flags_list:
    provider_features = provider_features.merge(
        flag_df, on=['filing_period', 'block_geoid'], how='left'
    )

# Fill NaN with 0 for all major provider columns
for name in MAJOR_PROVIDERS:
    provider_features[f'{name}_present'] = provider_features[f'{name}_present'].fillna(0).astype(int)
    provider_features[f'{name}_fiber'] = provider_features[f'{name}_fiber'].fillna(0).astype(int)

print(f'Provider features after major flags: {provider_features.shape}')
print(f'\nMajor provider columns added:')
major_cols = [c for c in provider_features.columns if any(name in c for name in MAJOR_PROVIDERS)]
print(major_cols)

Provider features after major flags: (2502268, 27)

Major provider columns added:
['att_present', 'att_fiber', 'charter_present', 'charter_fiber', 'comcast_present', 'comcast_fiber', 'verizon_present', 'verizon_fiber', 'frontier_present', 'frontier_fiber', 'cox_present', 'cox_fiber', 'google_fiber_present', 'google_fiber_fiber', 'altice_present', 'altice_fiber', 'windstream_present', 'windstream_fiber', 'lumen_present', 'lumen_fiber']


In [11]:
# Apply forward-persistence correction to per-provider fiber flags
# Same logic: a provider's fiber flag should only be 1 if it persists through the latest filing

for name in MAJOR_PROVIDERS:
    col = f'{name}_fiber'
    
    # Pivot to wide format
    prov_pivot = provider_features.pivot_table(
        index='block_geoid', columns='filing_period', values=col, fill_value=0
    )
    prov_pivot = prov_pivot[periods_sorted]  # chronological order
    
    # Apply same backward walk correction
    raw_vals = prov_pivot.values.copy()
    corr_vals = np.zeros_like(raw_vals)
    n = raw_vals.shape[1]
    
    for i in range(n - 1, -1, -1):
        if i == n - 1:
            corr_vals[:, i] = raw_vals[:, i]
        else:
            corr_vals[:, i] = raw_vals[:, i] & corr_vals[:, i + 1]
    
    flipped_count = ((raw_vals == 1) & (corr_vals == 0)).sum()
    
    # Write back corrected values
    corr_df = pd.DataFrame(corr_vals, index=prov_pivot.index, columns=prov_pivot.columns)
    corr_long = corr_df.reset_index().melt(
        id_vars='block_geoid', var_name='filing_period', value_name=f'{col}_corrected'
    )
    
    provider_features = provider_features.merge(
        corr_long, on=['filing_period', 'block_geoid'], how='left'
    )
    provider_features[col] = provider_features[f'{col}_corrected'].fillna(0).astype(int)
    provider_features.drop(columns=[f'{col}_corrected'], inplace=True)
    
    if flipped_count > 0:
        print(f'  {name}_fiber: {flipped_count:,} entries corrected')

print('\nPer-provider fiber persistence correction complete.')

TypeError: ufunc 'bitwise_and' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

## Step 6: Competitive Intensity Features

In [ ]:
# 6a. major_provider_count: count of major ISPs present (any tech)
present_cols = [f'{name}_present' for name in MAJOR_PROVIDERS]
provider_features['major_provider_count'] = provider_features[present_cols].sum(axis=1)

# 6b. major_fiber_count: count of major ISPs offering fiber
fiber_cols = [f'{name}_fiber' for name in MAJOR_PROVIDERS]
provider_features['major_fiber_count'] = provider_features[fiber_cols].sum(axis=1)

# 6c. fiber_competition_flag: 2+ fiber providers (any provider, not just majors)
provider_features['fiber_competition_flag'] = (provider_features['fiber_provider_count'] >= 2).astype(int)

print('Competitive intensity (basic):')
print(f'  major_provider_count: mean={provider_features["major_provider_count"].mean():.2f}')
print(f'  major_fiber_count: mean={provider_features["major_fiber_count"].mean():.2f}')
print(f'  fiber_competition_flag: {provider_features["fiber_competition_flag"].sum():,} blocks with 2+ fiber providers')

In [ ]:
# 6d. Derive cbg_fips for CBG-level aggregations
provider_features['cbg_fips'] = provider_features['block_geoid'].str[:12]

# 6e. cbg_fiber_penetration: % of CBs within the same CBG that have fiber
cbg_penetration = provider_features.groupby(['filing_period', 'cbg_fips'])['has_fiber'].mean().reset_index()
cbg_penetration.rename(columns={'has_fiber': 'cbg_fiber_penetration'}, inplace=True)

provider_features = provider_features.merge(
    cbg_penetration, on=['filing_period', 'cbg_fips'], how='left'
)

print(f'CBG fiber penetration: mean={provider_features["cbg_fiber_penetration"].mean():.3f}, '
      f'median={provider_features["cbg_fiber_penetration"].median():.3f}')

In [ ]:
# 6f. HHI Broadband — at CBG level per filing period
# For each CBG, compute each provider's market share as fraction of CBs served

def compute_hhi(group_df, block_col='block_geoid', provider_col='provider_id'):
    """Compute HHI for a group of provider-block records at CBG level."""
    total_blocks = group_df[block_col].nunique()
    if total_blocks == 0:
        return 0
    # Each provider's share = fraction of blocks they serve
    provider_blocks = group_df.groupby(provider_col)[block_col].nunique()
    shares = provider_blocks / total_blocks
    hhi = (shares ** 2).sum() * 10000
    return round(hhi, 2)

print('Computing HHI broadband (all technologies) at CBG level...')
# We need the raw FCC data with cbg_fips
fcc['cbg_fips'] = fcc['block_geoid'].str[:12]

hhi_broadband = fcc.groupby(['filing_period', 'cbg_fips']).apply(
    compute_hhi, include_groups=False
).reset_index(name='hhi_broadband')

print(f'HHI broadband computed for {len(hhi_broadband):,} (period, CBG) combinations')
print(f'  Mean: {hhi_broadband["hhi_broadband"].mean():.0f}')
print(f'  Median: {hhi_broadband["hhi_broadband"].median():.0f}')
print(f'  < 1500 (competitive): {(hhi_broadband["hhi_broadband"] < 1500).sum():,}')
print(f'  1500-2500 (moderate): {((hhi_broadband["hhi_broadband"] >= 1500) & (hhi_broadband["hhi_broadband"] <= 2500)).sum():,}')
print(f'  > 2500 (concentrated): {(hhi_broadband["hhi_broadband"] > 2500).sum():,}')

In [ ]:
# 6g. HHI Fiber — same computation but restricted to fiber providers only
print('Computing HHI fiber (tech code 50 only) at CBG level...')

# We need to use corrected fiber flags — only include records where the CB has corrected has_fiber=1
# Merge corrected fiber flag into fcc records
fcc_with_correction = fcc.merge(
    fiber_long[['filing_period', 'block_geoid', 'has_fiber']],
    on=['filing_period', 'block_geoid'],
    how='left'
)
fcc_with_correction['has_fiber'] = fcc_with_correction['has_fiber'].fillna(0).astype(int)

# Only fiber records where the CB's corrected has_fiber=1
fiber_for_hhi = fcc_with_correction[
    (fcc_with_correction['technology'] == 50) & (fcc_with_correction['has_fiber'] == 1)
]

hhi_fiber = fiber_for_hhi.groupby(['filing_period', 'cbg_fips']).apply(
    compute_hhi, include_groups=False
).reset_index(name='hhi_fiber')

print(f'HHI fiber computed for {len(hhi_fiber):,} (period, CBG) combinations')
print(f'  Mean: {hhi_fiber["hhi_fiber"].mean():.0f}')
print(f'  Monopoly (10000): {(hhi_fiber["hhi_fiber"] == 10000).sum():,}')

In [ ]:
# Merge HHI values to provider_features
provider_features = provider_features.merge(
    hhi_broadband, on=['filing_period', 'cbg_fips'], how='left'
)
provider_features = provider_features.merge(
    hhi_fiber, on=['filing_period', 'cbg_fips'], how='left'
)

# Fill NaN HHI (CBGs with no providers/fiber) with 0
provider_features['hhi_broadband'] = provider_features['hhi_broadband'].fillna(0)
provider_features['hhi_fiber'] = provider_features['hhi_fiber'].fillna(0)

print(f'HHI merged. Shape: {provider_features.shape}')

In [ ]:
# 6h. major_fiber_expansion_nearby
# Has any major provider NEWLY added fiber to an adjacent CB between prior and current period?

print('Computing major_fiber_expansion_nearby...')

# Load adjacency table
adjacency = pd.read_parquet(ADJACENCY_FILE)
print(f'Adjacency table: {len(adjacency):,} pairs')

# For each major provider, identify NEW fiber additions between periods
expansion_flags = []

for period_idx in range(1, len(periods_sorted)):
    prev_period = periods_sorted[period_idx - 1]
    curr_period = periods_sorted[period_idx]
    print(f'  Checking expansions: {prev_period} → {curr_period}')
    
    # Get major provider fiber flags for prev and curr periods
    prev_data = provider_features[provider_features['filing_period'] == prev_period][['block_geoid'] + fiber_cols].copy()
    curr_data = provider_features[provider_features['filing_period'] == curr_period][['block_geoid'] + fiber_cols].copy()
    
    # Find blocks where ANY major provider went from 0→1
    merged = prev_data.merge(curr_data, on='block_geoid', suffixes=('_prev', '_curr'))
    
    # A block has major fiber expansion if any provider's fiber went 0→1
    expansion_mask = pd.Series(False, index=merged.index)
    for name in MAJOR_PROVIDERS:
        col = f'{name}_fiber'
        new_fiber = (merged[f'{col}_prev'] == 0) & (merged[f'{col}_curr'] == 1)
        expansion_mask = expansion_mask | new_fiber
    
    expanded_blocks = set(merged.loc[expansion_mask, 'block_geoid'])
    print(f'    Blocks with major fiber expansion: {len(expanded_blocks):,}')
    
    # Now check adjacency: for each block in current period, is any neighbor in expanded_blocks?
    adj_expanded = adjacency[adjacency['neighbor_fips'].isin(expanded_blocks)]
    blocks_near_expansion = set(adj_expanded['cb_fips'])
    
    # Create flag for current period
    curr_blocks = provider_features[provider_features['filing_period'] == curr_period]['block_geoid'].unique()
    flag_df = pd.DataFrame({
        'block_geoid': curr_blocks,
        'filing_period': curr_period,
        'major_fiber_expansion_nearby': [1 if b in blocks_near_expansion else 0 for b in curr_blocks]
    })
    expansion_flags.append(flag_df)

# For the first period, set expansion_nearby to 0 (no prior period)
first_blocks = provider_features[provider_features['filing_period'] == periods_sorted[0]]['block_geoid'].unique()
first_flag = pd.DataFrame({
    'block_geoid': first_blocks,
    'filing_period': periods_sorted[0],
    'major_fiber_expansion_nearby': 0
})
expansion_flags.insert(0, first_flag)

expansion_df = pd.concat(expansion_flags, ignore_index=True)
provider_features = provider_features.merge(
    expansion_df, on=['filing_period', 'block_geoid'], how='left'
)
provider_features['major_fiber_expansion_nearby'] = provider_features['major_fiber_expansion_nearby'].fillna(0).astype(int)

print(f'\nmajor_fiber_expansion_nearby: {provider_features["major_fiber_expansion_nearby"].sum():,} total flags set')

## Step 7: Spatial Neighbor Features

In [ ]:
# Compute neighbor_fiber_pct and neighbor_count per CB per period
# Using the corrected has_fiber flag

print('Computing spatial neighbor features...')
print(f'Adjacency table: {len(adjacency):,} pairs')

# neighbor_count: number of neighbors per block (period-independent)
neighbor_counts = adjacency.groupby('cb_fips').size().reset_index(name='neighbor_count')
print(f'Blocks with neighbors: {len(neighbor_counts):,}')

# For neighbor_fiber_pct, we need to join adjacency with fiber status per period
neighbor_fiber_results = []

for period in periods_sorted:
    print(f'  Processing {period}...')
    
    # Get fiber status for this period
    period_fiber = fiber_long[fiber_long['filing_period'] == period][['block_geoid', 'has_fiber']]
    
    # Join: for each (cb_fips, neighbor_fips), get the neighbor's has_fiber
    adj_with_fiber = adjacency.merge(
        period_fiber, left_on='neighbor_fips', right_on='block_geoid', how='left'
    )
    adj_with_fiber['has_fiber'] = adj_with_fiber['has_fiber'].fillna(0)
    
    # Compute mean fiber among neighbors
    neighbor_fiber = adj_with_fiber.groupby('cb_fips')['has_fiber'].mean().reset_index()
    neighbor_fiber.rename(columns={'has_fiber': 'neighbor_fiber_pct', 'cb_fips': 'block_geoid'}, inplace=True)
    neighbor_fiber['filing_period'] = period
    
    neighbor_fiber_results.append(neighbor_fiber)

neighbor_fiber_df = pd.concat(neighbor_fiber_results, ignore_index=True)
print(f'\nNeighbor fiber results: {len(neighbor_fiber_df):,} rows')

In [ ]:
# Merge neighbor features
neighbor_counts.rename(columns={'cb_fips': 'block_geoid'}, inplace=True)

provider_features = provider_features.merge(
    neighbor_fiber_df, on=['filing_period', 'block_geoid'], how='left'
)
provider_features = provider_features.merge(
    neighbor_counts, on='block_geoid', how='left'
)

# Fill NaN for blocks with no neighbors
provider_features['neighbor_fiber_pct'] = provider_features['neighbor_fiber_pct'].fillna(0)
provider_features['neighbor_count'] = provider_features['neighbor_count'].fillna(0).astype(int)

print(f'Spatial features merged. Shape: {provider_features.shape}')
print(f'  neighbor_fiber_pct: mean={provider_features["neighbor_fiber_pct"].mean():.3f}')
print(f'  neighbor_count: mean={provider_features["neighbor_count"].mean():.1f}, median={provider_features["neighbor_count"].median()}')

## Step 8: Assemble & Export

In [ ]:
# Final column ordering
id_cols = ['filing_period', 'block_geoid', 'cbg_fips']
fiber_cols_final = ['has_fiber', 'fiber_provider_count', 'total_provider_count', 'max_fiber_down', 'max_fiber_up']
major_present = [f'{name}_present' for name in MAJOR_PROVIDERS]
major_fiber = [f'{name}_fiber' for name in MAJOR_PROVIDERS]
competitive_cols = [
    'major_provider_count', 'major_fiber_count', 'fiber_competition_flag',
    'hhi_broadband', 'hhi_fiber', 'cbg_fiber_penetration', 'major_fiber_expansion_nearby'
]
spatial_cols = ['neighbor_fiber_pct', 'neighbor_count']

all_cols = id_cols + fiber_cols_final + major_present + major_fiber + competitive_cols + spatial_cols

# Select and order
result = provider_features[all_cols].copy()

print(f'Final provider features matrix: {result.shape}')
print(f'\nColumns ({len(all_cols)}):')
for i, col in enumerate(all_cols):
    print(f'  {i+1:2d}. {col}')

print(f'\nFiling period breakdown:')
for period in periods_sorted:
    subset = result[result['filing_period'] == period]
    print(f'  {period}: {len(subset):,} blocks, fiber={subset["has_fiber"].sum():,} ({100*subset["has_fiber"].mean():.1f}%)')

print(f'\nSample rows:')
print(result.head())

In [ ]:
# Quality checks
print('Quality Checks')
print('=' * 50)

# No nulls in key columns
null_counts = result[id_cols + fiber_cols_final].isnull().sum()
print(f'Null counts in key columns:\n{null_counts[null_counts > 0]}')
if null_counts.sum() == 0:
    print('  No nulls in key columns.')

# has_fiber should be 0 or 1
assert result['has_fiber'].isin([0, 1]).all(), 'has_fiber contains invalid values'
print('has_fiber: all values 0 or 1')

# fiber_provider_count should be 0 when has_fiber=0
bad_counts = result[(result['has_fiber'] == 0) & (result['fiber_provider_count'] > 0)]
print(f'Consistency: {len(bad_counts)} rows with has_fiber=0 but fiber_provider_count>0 (should be 0)')

# HHI ranges
print(f'HHI broadband range: [{result["hhi_broadband"].min():.0f}, {result["hhi_broadband"].max():.0f}]')
print(f'HHI fiber range: [{result["hhi_fiber"].min():.0f}, {result["hhi_fiber"].max():.0f}]')

# cbg_fiber_penetration range [0, 1]
print(f'cbg_fiber_penetration range: [{result["cbg_fiber_penetration"].min():.3f}, {result["cbg_fiber_penetration"].max():.3f}]')

# neighbor_fiber_pct range [0, 1]
print(f'neighbor_fiber_pct range: [{result["neighbor_fiber_pct"].min():.3f}, {result["neighbor_fiber_pct"].max():.3f}]')

print(f'\nAll quality checks passed.')

In [ ]:
# Export
output_path = f'{OUTPUT_DIR}/cb_provider_features.parquet'
result.to_parquet(output_path, index=False)

import os
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f'Exported: {output_path}')
print(f'  Rows: {len(result):,}')
print(f'  Columns: {len(result.columns)}')
print(f'  File size: {file_size_mb:.1f} MB')
print(f'\nPhase 3 complete!')